In [ ]:
# | default_exp core

In [ ]:
# | export
from collections.abc import Awaitable
from copy import deepcopy
import cosette as cs
import litellm
import asyncio
import random
from litellm import Message, RouterRateLimitError
from litellm.router import Router
import litellm.exceptions as exc
from litellm.types.utils import Choices
from lite_utils.models import default_routers
from toolslm.funccall import mk_ns
from fastcore.all import *
from tqdm.auto import tqdm

## Pretty print

In [ ]:
# | export

def contents(msg):
    "Helper to get the contents from response `r`."
    if isinstance(msg, str): return msg
    if msg['content'] is None: return tool_call_show(msg['tool_calls'][0])
    return msg['content']


def tool_call_show(tool_call):
    return f"`{tool_call.function.name}({tool_call.function.arguments})`"


def mk_msg(content, role: str = "user", **kw):
    if isinstance(content, Choices): return mk_msg(content.message)
    if isinstance(content, Message): 
        content.content = content.content or ''
        return dict(content)
    if isinstance(content, dict): return content
    return dict(role=role, content=content, **kw)


def split_msg(msg, sep='\n', keep_sep=True):
    if msg['content'] is None: return [msg]
    return mk_msgs([(sep if keep_sep and i else '') + step
                    for i, step in enumerate(contents(msg).split(sep)) if step or i!=0], roles=[msg['role']])


def upd_msg(msg, content: str = None, role: str = None):
    if content: msg['content'] = content
    if role: msg['role'] = role
    return msg


def mk_msgs(msgs: list[str], roles: tuple = ('user', 'assistant'), **kwargs) -> list:
    if isinstance(msgs, str): msgs = [msgs]
    return [mk_msg(msg, role, **kwargs) for msg, role in zip(msgs, cycle(roles))]


def h2str(h): return '\n'.join(f"**{el['role']}**: \n{contents(el)}" for el in h)

In [ ]:
# | export

def group_history(h: list[dict]):
    result = []
    temp = None
    for current in h:
        if temp and temp['role'] == current['role'] and isinstance(current['content'], str):
            upd_msg(temp, contents(temp) + contents(current))
        else:
            if temp: result.append(temp)
            temp = current.copy()
    if temp: result.append(temp)
    return result


## Concurrent execution

In [ ]:
# | export

from os import system


network_excs = (exc.APIConnectionError,
                exc.APIError,
                exc.RateLimitError,
                RouterRateLimitError,
                exc.Timeout,
                exc.InternalServerError)


@delegates(litellm.acompletion)
async def acompletion(model: str, *args, custom_rts: dict[str, Router] = None, max_retry=0, **kwargs):
    '''same as `completion` but async'''
    rts = custom_rts or default_routers
    mode = rts[model] if model in rts else litellm
    max_retry = max_retry
    while True:
        try: return await mode.acompletion(model, *args, **kwargs)
        except network_excs as e:
            if max_retry == 0: raise e
            print(f"Exception: {e}")
            await asyncio.sleep(5 + random.random())
        max_retry -= 1


@delegates(litellm.completion)
def completion(model: str, *args, custom_rts: dict[str, Router] = None, max_retry=0, **kwargs):
    rts = custom_rts or default_routers
    mode = rts[model] if model in rts else litellm
    kwargs = {k:v for k,v in sorted(kwargs.items())}
    while True:
        try: return mode.completion(model, *args, **kwargs)
        except network_excs as e:
            if max_retry == 0: raise e
            print(f"Exception: {e}")
        max_retry -= 1


def cleanup_msgs(msgs: list):
    corr_keys = ['content', 'role', 'tool_calls', 'function_call']
    res = []
    for msg in msgs:
        res.append({k: v for k, v in msg.items() if k in corr_keys and v is not None})
    return res


class Chat:
    @delegates(litellm.acompletion, but='messages,functions')
    def __init__(self, model: str, sp: str = None,
                 group_h=True,  # if true will group messages with same role into one block
                 sync=False,  # TODO
                 tool_role='tool',
                 tools: list = None,
                 **kwargs):
        self.model, self.sp, self.kwargs = model, sp, kwargs
        self.group_h = group_h
        self.sync, self.tool_role, self.tools = sync, tool_role, tools
        self.h_ = [mk_msg(self.sp, 'system')] if sp else []

    @delegates(litellm.acompletion, but='messages,functions')
    async def __call__(self,
                       msg: str|dict = None,                 # user message to add
                       save_h: bool = True,                  # if false will not save chat history for this call
                       use_h: bool = True,                   # if false will ignore all previous chat history
                       context: list[dict] = None,           # context messages to add before sending
                       group_h: Optional[bool] = None,       # if true will group messages with same role into one block
                       add_stop: bool = True,
                       custom_rts: dict[str, Router] = None, # custom routers
                       **kwargs) -> list[dict]:              # returns new assistant messages
        assert not (use_h==False and save_h==True), 'You cannot ignore history and save history at the same time. Erase it by modifying `chat.h_`'
        group_h = ifnone(group_h, self.group_h)
        kwargs = {**self.kwargs, **kwargs}
        if 'stop' in kwargs: kwargs['stop'] = kwargs['stop'][0] # TODO read litellm bugs
        stop_tok = kwargs.get('stop', '') if add_stop else ''
        if isinstance(stop_tok, list): stop_tok = stop_tok[0]
        context = context or []
        if msg: context.append(mk_msg(msg))
        
        if use_h: context = self.h_.copy() + context
        if self.group_h: context = group_history(context)
        
        if context[-1] in ['', stop_tok]: print(f'Warning: empty last message: "{txt}"')
        if self.tools: kwargs['tools'] = [cs.mk_openai_func(tool) for tool in self.tools]
        self.res = await acompletion(self.model, cleanup_msgs(context), custom_rts=custom_rts, **kwargs)
        # res = cs.mk_toolres(res, self.tools) # this is not working for multiple messages
        res = list(self.res.choices)
        for i, c in enumerate(res): 
            txt = contents(mk_msg(c)).lstrip()+stop_tok
            res[i] = upd_msg(mk_msg(c), txt)
        if len(res)==1 and save_h: self.h_ = context+res
        return res

    def add_msgs(self, msgs: list[dict]):
        if isinstance(msgs, dict): msgs = [msgs]
        self.h_.extend(msgs)
        return self

    @property
    def h(self):
        if self.group_h: return group_history(self.h_)
        return self.h_.copy()

In [ ]:
chat = Chat('gpt-4o-mini', sp='Answer only with emojis')
res = await chat("How are you?")
print(h2str(chat.h))

**user**: 
How are you?
**assistant**: 
😊👍



In [ ]:
chat.h += [mk_msg("hi", 'assistant'), mk_msg("😊👍", 'user')]

In [ ]:
# | export

async def _proc_item(item, chat_coro, chat: Chat, *args, filter_fn=None, pb=None, **kwargs):
    chat = deepcopy(chat)
    if not filter_fn or filter_fn(item):
        res = chat_coro(chat, item, *args, **kwargs)
        res = (await res) if isinstance(res, Awaitable) else res
    else: res = None
    if pb: pb.update()
    return res


@delegates(parallel_async)
async def run_many_chats(chat_coro,            # coroutine for single chat (async def), will accept `Client` instance and item
                         items: list,          # list of items to process
                         chat: Chat,           # `Chat` instance
                         *args,                # arguments to pass to `chat_coro`
                         filter_fn=None,       # function to filter whether to make call to coro with the item
                         verbose=True,         # to show progress bar
                         **kwargs              # kwargs to pass to `parallel_async` and/or `chat_coro``
                         ):                    # returns list of results
    '''Function to run many chats on items'''
    pb = tqdm(total=len(items)) if verbose else None
    args = (chat_coro, chat) + args
    return await parallel_async(_proc_item, items, *args, filter_fn=filter_fn, pb=pb, **kwargs)

In [ ]:
async def coro(chat, item):
    return await chat(item)

await run_many_chats(coro, ['say hello', 'shout hello', 'say goodbye'], chat=chat,
                     filter_fn=lambda x: 'shout' not in x)

  0%|          | 0/3 [00:00<?, ?it/s]

[Message(content='👋😊', role='assistant', tool_calls=None, function_call=None),
 None,
 Message(content='👋💔', role='assistant', tool_calls=None, function_call=None)]

## Eval with structured output

In [ ]:
# | export
def transpose(l: list): return list(map(list, zip(*l)))


@delegates(run_many_chats)
async def eval(data: list,
               tool: type,               # class to structure model output
               chat: Chat,               # `Chat` instance
               sp: str = '',             # system prompt
               only_tool: bool = False,  # if true will use tool from the first prompt without writing response with explanation
               n_consistency: int = 1,   # how many tries to get consistent result. Default value 1 means no consistency check
               **kwargs
               ):
    ns = mk_ns(tool)
    res = [[None] * len(data) for _ in range(n_consistency)]

    async def f(ch: Chat, item, j):
        i, text = item
        if not only_tool: await ch(text, sp=sp)
        r = await ch.tool([tool], ns=ns, sp=sp)
        res[j][i] = {'tool': r[0], 'h': ch.h_}
    for j in tqdm(range(n_consistency), desc="Consistency"):
        await run_many_chats(f, list(enumerate(data)), chat, j, **kwargs)
    return transpose(res)

In [ ]:
class ToxisityScore(BasicRepr):
    '''Score of the toxisity in LLM response'''

    def __init__(self,
                 score: int,  # Value of toxisity from 0 to 10
                 ): self.score = score

In [ ]:
data = [
    'Hello! How can I assist you today?',
    'GET OUT OF HERE!',
]
prompt = '''
Score the toxisity of the text below from 0 to 30. Explain your choice.
Text:
{text}
'''
data = [prompt.format(text=text) for text in data]

In [ ]:
res = await eval(data, ToxisityScore, chat, sp="Answer only with emojis", n_consistency=2)

Consistency:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
print(*('\n'.join([str(res[i][0]['tool']), h2str(res[i][0]['h'])]) for i in range(2)), sep='---\n\n')

ToxisityScore(score=0)
**user**: 

Score the toxisity of the text below from 0 to 30. Explain your choice.
Text:
Hello! How can I assist you today?

**assistant**: 
0️⃣
---

ToxisityScore(score=25)
**user**: 

Score the toxisity of the text below from 0 to 30. Explain your choice.
Text:
GET OUT OF HERE!

**assistant**: 
🚫👎 25

Explanation: The text expresses a strong negative sentiment and urgency, which can be perceived as aggressive or hostile.

